In [1]:
import pandas as pd
from scipy.special.cython_special import xlog1py
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
data = pd.read_csv("../data/final_data.csv")
data.head(3)

,id,title,tagline
0,19995,Avatar,action adventure fantasy science fiction in th...
1,285,Pirates of the Caribbean: At World's End,"adventure fantasy action captain barbossa, lon..."
2,206647,Spectre,action adventure crime a cryptic message from ...


In [3]:
data["title"].duplicated().sum()

np.int64(0)

In [4]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()
data["tagline"] = data["tagline"].apply(lambda x : ps.stem(x))

In [5]:
data["tagline"][0]

'action adventure fantasy science fiction in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. sam worthington zoe saldana sigourney weaver james cameron'

In [6]:
vectorizer = CountVectorizer(max_features=5000, stop_words="english")

X1 = vectorizer.fit_transform(data["tagline"]).toarray()

In [7]:
vectorizer.get_feature_names_out()

array(['000', '007', '10', ..., 'zooey', 'zuck', 'zwick'],
      shape=(5000,), dtype=object)

In [8]:
similarity = cosine_similarity(X1)

In [9]:
similarity.shape

(4797, 4797)

In [10]:
import pickle

pickle.dump(similarity, open("../model/similarity_distance.pkl", "wb"))
pickle.dump(data.to_dict(), open("../model/final_data.pkl", "wb"))

In [11]:
def movie_recommend(name: str) -> list:
    matching_movies = data.loc[data["title"] == name]

    if matching_movies.empty:
        return [f"Sorry, '{name}' was not found in the database."]

    movie_index = matching_movies.index[0]
    distance = similarity[movie_index]

    movie_list = sorted(list(enumerate(distance)), key= lambda x : x[1], reverse=True)

    recommended_movie = []
    for val in movie_list[1:6]:
        movie_title = data.iloc[val[0]].title
        recommended_movie.append(movie_title)
    return recommended_movie

In [12]:
result = movie_recommend("Avatar")
result

['Guardians of the Galaxy',
 'Aliens',
 'The Book of Life',
 'Godzilla 2000',
 'Small Soldiers']